In [19]:
import torch 
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer 
from peft import get_peft_model, LoraConfig
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


The CorDA tuning step looked at how to tune a CorDA model efficiently using hugging face's high level tuning library and training loops. In this notebook, I will demonstrate what happens under the hood when tuning a smaller model -- for testing purposes -- and as well as the effectiveness of adapter tuning on downstream tasks. 

### Load the gpt-neo-125M model 

In [20]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125m")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")
print(model)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [21]:
GPT_NEO_BLOCK = model.transformer.h
attention_block = []

for gpt_block in GPT_NEO_BLOCK:
    attention_block.append(gpt_block.attn.attention)

print(attention_block[0])

GPTNeoSelfAttention(
  (attn_dropout): Dropout(p=0.0, inplace=False)
  (resid_dropout): Dropout(p=0.0, inplace=False)
  (k_proj): Linear(in_features=768, out_features=768, bias=False)
  (v_proj): Linear(in_features=768, out_features=768, bias=False)
  (q_proj): Linear(in_features=768, out_features=768, bias=False)
  (out_proj): Linear(in_features=768, out_features=768, bias=True)
)


In [4]:
#We'll use an example attention head to demonstrate the Linear layers we're performing Lora on 
attention_1 = attention_block[0]
k_proj = attention_1.k_proj
## Print the raw tensor of the K proj matrix and confirm it's shape 
print(k_proj.weight.shape)
print(k_proj.weight.dtype)

torch.Size([768, 768])
torch.float32


### Save model's old params locally

We'll save the params of each block's q, k, v projections 

In [5]:
import torch as pt
import os

EXPERIMENT_NUMBER = 0
proj_modules = ["k_proj", "q_proj", "v_proj"]

for idx, attention in enumerate(attention_block):
    block_dir = f"../model/lora/base_model_attention_proj_weights/block_{idx}"
    os.makedirs(block_dir, exist_ok=True)
    pt.save(attention.k_proj.weight.detach().cpu(), f"{block_dir}/k_proj.pt")
    pt.save(attention.q_proj.weight.detach().cpu(), f"{block_dir}/q_proj.pt")
    pt.save(attention.v_proj.weight.detach().cpu(), f"{block_dir}/v_proj.pt")


### Define LoRa Initializer

In [22]:
class LoRaHelper(nn.Module):
    def __init__(self, base_layer: nn.Linear, alpha:int , in_dims: int = 768, r:int = 4 ):
        super().__init__()
        
        self.base = base_layer
        self.base.weight.requires_grad = False
        
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
            
        in_dims = self.base.in_features
        out_dims = self.base.out_features
            
        self.alpha = alpha 
        self.r = r 
        self.scaling = self.alpha / self.r
        
        # Up projection layer: r x k
        self.init_A(r = r , k = in_dims)
        # Down projection layer: d x r 
        self.init_B(r=r, d= out_dims)
        
        #.Parameter subclasses the tensor class and tell us that these tensor are learnable params inside of the nn.Module
        self.A = nn.Parameter(self.A)
        self.B = nn.Parameter(self.B)
         
    def forward(self, x):
        result = self.base(x)
        
        down_proj = x @ self.A.T
        
        #up_proj = BAx
        up_proj = down_proj @ self.B.T
        
        h = result + self.scaling * up_proj 
        return h 
        
    # A has dims R ^ {r * k }
    def init_A(self, r: int, k:int ):
        self.A = torch.empty(r, k)
        self.A = torch.nn.init.normal_(self.A, mean=0.0, std=0.02)
        
    def init_B(self, r:int, d:int):
        self.B = torch.empty(d,r)
        

### Define Hyper params 

In [23]:
EPOCHS = 5 
alpha = 0.02
lr = 0.01
batch_size = 2 

### Load the dataset 

In [24]:
from datasets import load_dataset
from utils.dataset import getRawDataset, preprocess
import importlib

# importlib.reload(dataset)
# importlib.reload(text_cleaners)

BATCH_SIZE = 5
datasets_to_test = ["wikitext", "imdb", "sst2", "squad_v2"]

for dataset_name in datasets_to_test:
    print(f"\n=== {dataset_name.upper()} ===")
    
    raw_ds = getRawDataset(dataset_name, split="train")
    
    raw_batch = raw_ds.select(range(BATCH_SIZE))
    
    print("\n--- RAW ---")
    for i, item in enumerate(raw_batch):
        print(f"{i}: {item}")
    
    batch_dict = raw_batch[:]
    # if dataset_name.lower() == 'sst2':
    #     # rename 'sentence' -> 'text' so _preprocessSST2 can work as other datasets
    #     batch_dict = {'text': batch_dict['sentence'], 'label': batch_dict['label']}

    cleaned_batch = preprocess(batch_dict, dataset_name)
    print("\n--- CLEANED ---")
    batch_len = len(next(iter(cleaned_batch.values())))
    for i in range(batch_len):
        cleaned_item = {k: cleaned_batch[k][i] for k in cleaned_batch}
        
        if 'text' in cleaned_item:
            print(f"{i}: {cleaned_item['text']}")
        elif 'sentence' in cleaned_item:
            print(f"{i}: {cleaned_item['sentence']}")
        elif 'context' in cleaned_item:
            print(f"{i}: {cleaned_item['context']}")
        else:
            print(f"{i}: {cleaned_item}")



=== WIKITEXT ===
Loaded wikitext (train): 36718

--- RAW ---
0: {'text': ''}
1: {'text': ' = Valkyria Chronicles III = \n'}
2: {'text': ''}
3: {'text': ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'}
4: {'text': " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it r

Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 916356.04 examples/s]


Loaded sst2 (train): 67349

--- RAW ---
0: {'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
1: {'sentence': 'contains no wit , only labored gags ', 'label': 0, 'idx': 1}
2: {'sentence': 'that loves its characters and communicates something rather beautiful about human nature ', 'label': 1, 'idx': 2}
3: {'sentence': 'remains utterly satisfied to remain the same throughout ', 'label': 0, 'idx': 3}
4: {'sentence': 'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ', 'label': 0, 'idx': 4}

--- CLEANED ---
0: hide new secretions from the parental units
1: contains no wit , only labored gags
2: that loves its characters and communicates something rather beautiful about human nature
3: remains utterly satisfied to remain the same throughout
4: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up

=== SQUAD_V2 ===


Generating validation split: 100%|██████████| 11873/11873 [00:00<00:00, 1638878.81 examples/s]

Loaded squad_v2 (train): 130319

--- RAW ---
0: {'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'question': 'When did Beyonce start becoming popular?', 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}
1: {'id': '56be85543aeaaa14008c9065', 'title': 'Beyoncé', 'con

### Initialize the LoRa Helper Class for each linear projection head

In [25]:
for attention_head in attention_block:
    attention_head.k_proj = LoRaHelper(
        base_layer= attention_head.k_proj,
        alpha=16
    )

### Create tuning loop

In [26]:
from utils import dataset

tokenizer.pad_token = tokenizer.eos_token

# To Optimize training only use the optimizer on trainable params in the network
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad]
    ,lr=1e-4)

loss_fn = torch.nn.CrossEntropyLoss()

datasets= ['wikitext']

#Set model to GPU 
model.to(device)

for dataset_name in datasets:
    ds = dataset.getRawDataset(dataset_name, "train")
    ds = ds.with_format("torch")
    dataloader = DataLoader(ds,batch_size=batch_size, shuffle=True, drop_last=True) 

    for epoch in range(EPOCHS):    
        model.train()

        for step, batch in enumerate(tqdm(dataloader)):      

            cleaned = dataset.preprocess(batch, dataset_name)    

            # Determine which field to tokenize based on dataset type
            if dataset_name.lower() == "squad_v2":
                # SQuAD requires both context and question
                # You can concatenate them with a separator or feed them separately
                tokenized_batch = tokenizer(
                    text=cleaned['question'],
                    text_pair=cleaned['context'],
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                    max_length=384
                ).to(model.device)
            else:
                if len(cleaned['text']) == 0:
                    continue
                # Standard text-based datasets
                tokenized_batch = tokenizer(
                    cleaned['text'],
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                    max_length=96
                ).to(model.device)

            input_ids = tokenized_batch["input_ids"].to(model.device)
            attention_mask = tokenized_batch["attention_mask"].to(model.device)
            
            # This is the forward pass
            preds = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids) 
            logits = preds["logits"]
            loss = preds.loss
            
            # # zero out the gradients 
            optimizer.zero_grad()
            # #Run back prop once for the step 
            loss.backward()
            # #Use the defined optimizer to update the grads accordingly 
            # # Tells the optimizer to take a step in gradient descent and update the grads with the provided learning rate
            optimizer.step()
            
            if step % 50 == 0:
                print(f"epoch {epoch} step {step} loss: {loss.item()}" )
        
        

Loaded wikitext (train): 36718


  0%|          | 58/18359 [00:01<09:09, 33.31it/s]

epoch 0 step 50 loss: nan


  1%|          | 108/18359 [00:03<09:21, 32.52it/s]

epoch 0 step 100 loss: nan


...

epoch 0 step 200 loss: nan


  1%|▏         | 256/18359 [00:07<08:27, 35.65it/s]

epoch 0 step 250 loss: nan


  2%|▏         | 288/18359 [00:08<08:54, 33.84it/s]


KeyboardInterrupt: 